#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [2]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive


#`Setup And Update ComfyUI`



In [3]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 61219, done.
remote: Counting objects: 100% (521/521), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 61219 (delta 469), reused 393 (delta 393), pack-reused 60698 (from 3)
Receiving objects: 100% (61219/61219), 97.04 MiB | 17.63 MiB/s, done.
Resolving deltas: 100% (42337/42337), done.
/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu118, https://download.pytorch.org/whl/cu117
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# `Models Download`

## Download MODELS (manifest các model đang dùng — HF direct)

In [5]:
#@title Download model đang dùng — Hugging Face direct (verified)
%cd {WORKSPACE}
# Install Aria2
!apt-get -y install -qq aria2

import os
from google.colab import userdata

# Nhóm model đang dùng — đặt True/False cho nhóm cần tải (~76G nếu bật tất!)
WANT = {
  "qwen-image":    True,   # ~28.7G — Qwen-Image-2512 fp8 + qwen2.5-vl 7B encoder + VAE + Lightning LoRA
  "trellis":       True,   # ~6.8G  — TRELLIS.2 int8 + shape/texture VAE
  "pixal3d":       True,   # ~6.3G  — Pixal3D int8 + dino_v3 clip_vision
  "3d-aux":        True,   # ~9.7G  — qwen3vl 8B encoder + birefnet + moge2 (aux cho TRELLIS.2/Pixal3D)
  "acestep":       True,   # ~9.4G  — ACE-Step 1.5 turbo + qwen 0.6B/1.7B encoders + VAE
  "minimax-music": True,   # ~4.8G  — MiniMax Music 3 DiT fp16 + dav VAE
  "stable-audio":  True,   # ~9.7G  — Stable Audio 3 medium + t5gemma encoder
  "qwen3.5":       False,  # ~4.2G  — qwen3.5 2B bf16 text encoder
}

# (group, models/<subdir>, filename, url, ~GB)
MODELS = [
  ("qwen-image", "diffusion_models", "qwen_image_2512_fp8_e4m3fn.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors", 19.0),
  ("qwen-image", "text_encoders", "qwen_2.5_vl_7b_fp8_scaled.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors", 8.7),
  ("qwen-image", "vae", "qwen_image_vae.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors", 0.2),
  ("qwen-image", "loras", "Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors",
   "https://huggingface.co/lightx2v/Qwen-Image-2512-Lightning/resolve/main/Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors", 0.8),

  ("trellis", "diffusion_models", "trellis_2_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/diffusion_models/trellis_2_int8_convrot.safetensors", 4.9),
  ("trellis", "vae", "trellis_2_shape_vae_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/vae/trellis_2_shape_vae_bf16.safetensors", 1.0),
  ("trellis", "vae", "trellis_2_texture_vae_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/vae/trellis_2_texture_vae_bf16.safetensors", 0.9),

  ("pixal3d", "diffusion_models", "pixal3d_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/Pixal3D/resolve/main/diffusion_models/pixal3d_int8_convrot.safetensors", 5.2),
  ("pixal3d", "clip_vision", "dino_v3_L_naf_fp32.safetensors",
   "https://huggingface.co/Comfy-Org/Pixal3D/resolve/main/clip_vision/dino_v3_L_naf_fp32.safetensors", 1.1),

  ("3d-aux", "text_encoders", "qwen3vl_8b_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors", 8.7),
  ("3d-aux", "background_removal", "birefnet.safetensors",
   "https://huggingface.co/Comfy-Org/BiRefNet/resolve/main/background_removal/birefnet.safetensors", 0.4),
  ("3d-aux", "geometry_estimation", "moge_2_vitl_normal_fp16.safetensors",
   "https://huggingface.co/Comfy-Org/MoGe/resolve/main/geometry_estimation/moge_2_vitl_normal_fp16.safetensors", 0.6),

  ("acestep", "diffusion_models", "acestep_v1.5_turbo.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/diffusion_models/acestep_v1.5_turbo.safetensors", 4.5),
  ("acestep", "text_encoders", "qwen_0.6b_ace15.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_0.6b_ace15.safetensors", 1.1),
  ("acestep", "text_encoders", "qwen_1.7b_ace15.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_1.7b_ace15.safetensors", 3.5),
  ("acestep", "vae", "ace_1.5_vae.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/vae/ace_1.5_vae.safetensors", 0.3),

  ("minimax-music", "diffusion_models", "minimax_music3_dit_fp16.safetensors",
   "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main/diffusion_models/minimax_music3_dit_fp16.safetensors", 4.6),
  ("minimax-music", "vae", "minimax_music3_dav.safetensors",
   "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main/vae/minimax_music3_dav.safetensors", 0.2),

  ("stable-audio", "checkpoints", "stable_audio_3_medium.safetensors",
   "https://huggingface.co/Comfy-Org/stable-audio-3/resolve/main/checkpoints/stable_audio_3_medium.safetensors", 8.6),
  ("stable-audio", "text_encoders", "t5gemma_b_b_ul2.safetensors",
   "https://huggingface.co/Comfy-Org/stable-audio-3/resolve/main/text_encoders/t5gemma_b_b_ul2.safetensors", 1.1),

  ("qwen3.5", "text_encoders", "qwen3.5_2b_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen3.5/resolve/main/text_encoders/qwen3.5_2b_bf16.safetensors", 4.2),
]


def downloadModel(url, destdir, filename=None):
  dest = f"{WORKSPACE}/models/{destdir}"
  os.makedirs(dest, exist_ok=True)
  if filename is None:
    filename = url.split('/')[-1].removesuffix('?download=true')
  if os.path.exists(f"{dest}/{filename}"):
    print("skip (đã có):", filename)
    return
  !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest}" -o "{filename}"


selected = [m for m in MODELS if WANT.get(m[0], False)]
total = sum(m[4] for m in selected)
print(f"Sẽ tải {len(selected)} file (~{total:.1f} GB). Dùng DRIVE_PATH trên Drive để khỏi tải lại sau khi runtime reset.\n")

for group, destdir, filename, url, gb in selected:
  print(f"[{group}] {filename} (~{gb}G) -> models/{destdir}")
  downloadModel(url, destdir, filename)

# Tải thêm link bất kỳ (huggingface/civitai):
# downloadModel('https://civitai.com/api/download/models/2071650', 'checkpoints')  # cần CIVITAI_API_TOKEN
# downloadModel('https://huggingface.co/<repo>/resolve/main/model.safetensors', 'checkpoints')

# Custom nodes (uncomment để cài):
# !git clone https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager
# !git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack


/content/ComfyUI
Sẽ tải 20 file (~75.4 GB). Dùng DRIVE_PATH trên Drive để khỏi tải lại sau khi runtime reset.

[qwen-image] qwen_image_2512_fp8_e4m3fn.safetensors (~19.0G) -> models/diffusion_models
 *** Download Progress Summary as of Wed Oct 21 15:14:23 2026 *** 
=
[#e4fc82 8.7GiB/19GiB(46%) CN:16 DL:152MiB ETA:1m8s]
FILE: /content/ComfyUI/models/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors
-

 *** Download Progress Summary as of Wed Oct 21 15:15:24 2026 *** 
=
[#e4fc82 17GiB/19GiB(92%) CN:16 DL:140MiB ETA:9s]
FILE: /content/ComfyUI/models/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e4fc82|OK  |   152MiB/s|/content/ComfyUI/models/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors

Status Legend:
(OK):download completed.
[qwen-image] qwen_2.5_vl_7b_fp8_scaled.safetensors (~8.7G) -> models/text_encoders

Download Results:
gid

## LIST MODELS

In [6]:
!find ./models -type f -not -name 'put_*' -exec du -h {} + | sort -rh | head -40


20G	./models/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors
8.8G	./models/text_encoders/qwen3vl_8b_int8_convrot.safetensors
8.8G	./models/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors
8.6G	./models/checkpoints/stable_audio_3_medium.safetensors
5.3G	./models/diffusion_models/pixal3d_int8_convrot.safetensors
4.9G	./models/diffusion_models/trellis_2_int8_convrot.safetensors
4.6G	./models/diffusion_models/minimax_music3_dit_fp16.safetensors
4.5G	./models/diffusion_models/acestep_v1.5_turbo.safetensors
3.5G	./models/text_encoders/qwen_1.7b_ace15.safetensors
1.2G	./models/text_encoders/t5gemma_b_b_ul2.safetensors
1.2G	./models/text_encoders/qwen_0.6b_ace15.safetensors
1.2G	./models/clip_vision/dino_v3_L_naf_fp32.safetensors
1.1G	./models/vae/trellis_2_shape_vae_bf16.safetensors
905M	./models/vae/trellis_2_texture_vae_bf16.safetensors
811M	./models/loras/Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors
632M	./models/geometry_estimation/moge_2_vitl_normal_fp16.safetensors


# `START ComfyUI  & Expose Server (MANUAL)`

## Install cloudflared

In [7]:
# Cài cloudflared từ repo chính thức của Cloudflare (Debian/Ubuntu)
import os

SUDO = "" if os.geteuid() == 0 else "sudo"

!{SUDO} mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | {SUDO} tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | {SUDO} tee /etc/apt/sources.list.d/cloudflared.list
!{SUDO} apt-get update && {SUDO} apt-get install -y cloudflared
!cloudflared --version


deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 https://pkg.cloudflare.com/cloudflared any InRelease [5,039 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,957 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,170 kB]
Get:12 http://archive.ubuntu.com/ub

In [ ]:
!python main.py --dont-print-server

# `START ComfyUI & Expose Server`

## Install cloudflared

In [8]:
# Cài cloudflared từ repo chính thức của Cloudflare (Debian/Ubuntu)
import os

SUDO = "" if os.geteuid() == 0 else "sudo"

!{SUDO} mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | {SUDO} tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | {SUDO} tee /etc/apt/sources.list.d/cloudflared.list
!{SUDO} apt-get update && {SUDO} apt-get install -y cloudflared
!cloudflared --version


deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:8 https://pkg.cloudflare.com/cloudflared any InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Ign:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Ign:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/p

## CF Tunnel

In [ ]:
# Cloudflare named tunnel (không dùng quick tunnel / trycloudflare.com)
import subprocess
import threading
import time
import socket

from google.colab import userdata

CLOUDFLARE_TUNNEL_TOKEN = userdata.get('CLOUDFLARE_TUNNEL_TOKEN')
if not CLOUDFLARE_TUNNEL_TOKEN:
  raise ValueError(
      "Thiếu Colab secret 'CLOUDFLARE_TUNNEL_TOKEN'. Trong Cloudflare Zero Trust -> Networks -> "
      "Tunnels -> <tunnel của bạn> -> 'Install and run a connector' -> copy token."
  )

TUNNEL_HOSTNAME = "colab.fluxvane.com"  # @param {type:"string"}  # hostname public đã map trong Cloudflare, vd: comfy.example.com
PORT = 5173


def wait_port(port, timeout=600):
  deadline = time.time() + timeout
  while time.time() < deadline:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
      sock.settimeout(1)
      if sock.connect_ex(('127.0.0.1', port)) == 0:
        return True
    time.sleep(0.5)
  return False


def tunnel_thread(port):
  if not wait_port(port):
    print("\nComfyUI chưa sẵn sàng sau 10 phút - tunnel không được khởi động.\n")
    return
  print("\nComfyUI đã sẵn sàng, khởi động Cloudflare Tunnel (named tunnel)...\n")
  p = subprocess.Popen(
      ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN],
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in p.stdout:
    l = line.decode(errors='replace')
    if "Registered tunnel connection" in l:
      print("Tunnel đã đăng ký. Mở ComfyUI tại: https://" +
            (TUNNEL_HOSTNAME or "<hostname đã cấu hình trong Cloudflare>"))
    print(l, end='')


threading.Thread(target=tunnel_thread, daemon=True, args=(PORT,)).start()

%cd {WORKSPACE}
!python main.py --dont-print-server --port {PORT}

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'convrot_w4a4_linear', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_embedding', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'dequantize_w4a8_int8_weight', 'fp16_conv3d',